In [10]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
import os
import requests


In [26]:
@tool
def get_weather(city:str) -> str:
    """weather for city that is given"""
    api_key = os.getenv('WEATHER_API_KEY')
    url = f"https://api.weatherapi.com/v1/current.json?key={api_key}&q={city}"
    response = requests.get(url)
    return response.json()

@tool
def calculate(num:int, num2:int) -> int:
    """returns the first number to the power of second number"""
    return num**num2

In [28]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=.7,
    api_version='v1beta'
)
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt='you are helpful assistant that uses tools when needed',
    checkpointer=InMemorySaver()
)


In [29]:
config = {"configurable": {"thread_id": "session-1"}}

r = agent.invoke(
    {"messages": [{"role": "user", "content": "whats weather in kathmandu and 3 to the power of 2?"}]},
    config,
)
print(r["messages"][-1].content)

[{'type': 'text', 'text': 'The weather in Kathmandu is 14.1°C with smoky haze. 3 to the power of 2 is 9.', 'extras': {'signature': 'CpoCAWkUfRMp1afHpjntVFoiJx7NTb9pVTHgAhlWSU1Sqh6FZLDr1LrK3+oPrg2iJ2an6XPcuBgeRc+stK9Xw2Kc6uoQXlNASt26YLi7eJxc5G/4zWsVeGEJTZs8D0i5kMQb9nXeF+SuZXznFkthJ521n0zwLg9q0DNNAR/8X6QbITReo05d0O7zvHPpWKFzp00xBaWTNyTh2nbutS9bglm+PvYnWDyd9Rl9FfDOpthLJc1fjck+2ySG2kV9+EF2zCxR0B6e1hNqMcZSx4g8S/sPEOvr2rKHBdg+aZd0tkilLcmmAWO+ThP+EB3GtEoSLImMCPuMR0sZkPEkpS0Cbrp9/jFJ2ZmW24L7ge3s+va847Wk5keHgVdHPXkG'}}]
